# 11 — AutoML Experiments (FLAML)

**Objective:** Benchmark automated machine learning using Microsoft FLAML under a strict time budget.


In [1]:
import sys
import json
import time
from pathlib import Path
import pandas as pd
import numpy as np
from flaml import AutoML

def find_root():
    p = Path.cwd()
    for parent in [p] + list(p.parents):
        if (parent / 'data' / 'raw' / 'gt_2011.csv').exists():
            return parent
    return p

ROOT = find_root()
sys.path.insert(0, str(ROOT))

from src.data.loader import FEATURES_A
from src.data.splitting import temporal_split


D:\TURBINE_X_Full_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Executing Fast FLAML AutoML Run


In [2]:
df = pd.read_parquet(ROOT / "data" / "processed" / "cleaned_master.parquet")
dev, cal, test = temporal_split(df)

# Run FLAML on primary emission target CO
automl = AutoML()
settings = {
    "time_budget": 15,
    "metric": "rmse",
    "task": "regression",
    "seed": 42,
    "verbose": 0
}

t0 = time.time()
automl.fit(X_train=dev[FEATURES_A], y_train=dev["CO"], **settings)
elapsed = time.time() - t0

summary = {
    "best_estimator": str(automl.best_estimator),
    "best_loss_rmse": float(automl.best_loss),
    "time_to_find_best_s": float(automl.time_to_find_best_model),
    "total_time_s": float(elapsed)
}
print(json.dumps(summary, indent=2))

out_automl = ROOT / "experiments" / "automl_summary.json"
with open(out_automl, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"Saved AutoML summary to {out_automl.relative_to(ROOT)}")
print("AutoML experiments COMPLETE.")


{
  "best_estimator": "lgbm",
  "best_loss_rmse": 1.2299791162019165,
  "time_to_find_best_s": 13.270406723022461,
  "total_time_s": 15.221664428710938
}
Saved AutoML summary to experiments\automl_summary.json
AutoML experiments COMPLETE.
